In [1]:
library(tidyverse)

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
gg <- read_tsv("mm10genes.gtf", col_names = F)
glimpse(gg)

Rows: 48440 Columns: 9
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (7): X1, X2, X3, X6, X7, X8, X9
dbl (2): X4, X5

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 48,440
Columns: 9
$ X1 <chr> "chr1", "chr1", "chr1", "chr1", "chr1", "chr1", "chr1", "chr1", "ch…
$ X2 <chr> "HAVANA", "ENSEMBL", "HAVANA", "HAVANA", "HAVANA", "HAVANA", "HAVAN…
$ X3 <chr> "gene", "gene", "gene", "gene", "gene", "gene", "gene", "gene", "ge…
$ X4 <dbl> 3073253, 3102016, 3205901, 3252757, 3365731, 3375556, 3464977, 3466…
$ X5 <dbl> 3074322, 3102125, 3671498, 3253236, 3368549, 3377788, 3467285, 3513…
$ X6 <chr> ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".…
$ X7 <chr> "+", "+", "-", "+", "-", "-", "-", "+", "-", "+", "-", "-", "+", "+…
$ X8 <chr> ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".…
$ X9 <chr> "gene_id \"ENSMUSG00000102693.1\"; gene_type \"TEC\"; gene_status \…


In [3]:
gg <- gg |> mutate(tss = case_when(X7 == "+" ~ X4, .default = X5)) |> mutate(pstart = tss-500, pend =tss+500)
pbed <- gg |> select(X1,pstart,pend) |> arrange(X1,pstart)
nrow(pbed)

[1] 48440

In [4]:
write_tsv(pbed,"tmp.bed", col_names = F)
system("bedtools merge -i tmp.bed > mtmp.bed", wait = T)

In [5]:
mbed <- read_tsv("mtmp.bed",col_names = F)
nrow(mbed)

Rows: 40207 Columns: 3
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (1): X1
dbl (2): X2, X3

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] 40207

In [6]:
mbed$id <- "promoter"
mbed$id <- make.unique(mbed$id)
mbed <- arrange(mbed,X1,X2)
write_tsv(mbed, "promoters_mm10.bed", col_names = F)
system("rm *tmp.bed")